# Ingesta de Datos en Hadoop (HDFS) con PySpark

**Objetivo:** Leer el dataset `ml_dataset.csv` en bruto, usar PySpark para aplicar la limpieza necesaria y persistir los datos en HDFS (Hadoop Distributed File System) construyendo la capa base de nuestro Data Lake.

**HDFS paths:**  
- Raw  → `hdfs://hadoop:9000/data/raw/transactions.parquet`  
- Processed → `hdfs://hadoop:9000/data/processed/transactions_clean.parquet`  

> Este notebook se ejecuta desde **JupyterLab** (`ml-env`, puerto 8888).  
> El contenedor Hadoop debe estar levantado: `docker compose -f infrastructure/hadoop/docker-compose.yml up -d`

## 1. Instalación de Dependencias

In [1]:
# Solo necesario la primera vez en el entorno ml-env
import subprocess, sys

pkgs = ["pyspark==3.5.3", "pyarrow==16.1.0", "datasets==2.20.0", "huggingface_hub==0.23.4"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pkg], check=True)

print("Dependencias instaladas correctamente.")

Dependencias instaladas correctamente.


## 2. Configurar rutas para el Dataset local (PaySim)

In [2]:
# Usamos directamente la ruta montada por Docker apuntando a nuestro CSV original
LOCAL_CSV_PATH = "/app/data/raw/ml_dataset.csv"

print(f"Ruta del dataset origen configurada: {LOCAL_CSV_PATH}")

Ruta del dataset origen configurada: /app/data/raw/ml_dataset.csv


## 3. Inicializar SparkSession conectada a HDFS

In [3]:
import os
import socket
from pyspark.sql import SparkSession

# --- Detección automática de HDFS ---
HDFS_URI = "hdfs://hadoop:9000"
LOCAL_DATA = "/app/data"   # volumen montado en ml-env → ../../data

def _hdfs_available(host="hadoop", port=9000, timeout=4):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False

HDFS_UP = _hdfs_available()
if HDFS_UP:
    DATA_ROOT = HDFS_URI
    print(f"[OK] HDFS disponible → almacenamiento en {DATA_ROOT}")
else:
    DATA_ROOT = f"file://{LOCAL_DATA}"
    print(f"[WARN] Hadoop no detectado → modo local: {DATA_ROOT}")
    print("       Para usar HDFS: docker compose -f infrastructure/hadoop/docker-compose.yml up -d")

# --- SparkSession ---
builder = (
    SparkSession.builder
    .appName("FraudDetection-Ingesta")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    # 4 g para el driver: necesario para shuffle/features sobre 5 M filas en local[*]
    .config("spark.driver.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    # Arrow: pandas→Spark vectorizado (se usa en celdas de features)
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    # Evitar OOM en operaciones de sort/join grandes
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "50000")
)

if HDFS_UP:
    builder = (
        builder
        .config("spark.hadoop.fs.defaultFS", HDFS_URI)
        .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
        .config("spark.hadoop.ipc.client.connect.timeout", "10000")
        .config("spark.hadoop.ipc.client.connect.max.retries.on.timeouts", "3")
        .config("spark.hadoop.dfs.client.socket-timeout", "30000")
    )

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} iniciado  |  DATA_ROOT = {DATA_ROOT}")


[OK] HDFS disponible → almacenamiento en hdfs://hadoop:9000


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 16:08:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3 iniciado  |  DATA_ROOT = hdfs://hadoop:9000


## 4. Leer CSV nativo en Spark y guardar en HDFS (raw)

In [4]:
import os

# 1. Leer CSV nativamente en Big Data (Spark)
# file:// para forzar la lectura del volumen local del contenedor, no de HDFS
print(f"Leyendo CSV desde {LOCAL_CSV_PATH}...")
sdf_raw = spark.read.csv(f"file://{LOCAL_CSV_PATH}", header=True, inferSchema=True)

print(f"Spark DataFrame cargado: {sdf_raw.count():,} filas, {len(sdf_raw.columns)} columnas")
sdf_raw.printSchema()

# 2. Guardar raw en HDFS (formato columnar óptimo)
RAW_PATH = f"{DATA_ROOT}/raw/transactions.parquet"
if not HDFS_UP:
    os.makedirs(f"{LOCAL_DATA}/raw", exist_ok=True)

sdf_raw.write.mode("overwrite").parquet(RAW_PATH)
print(f"[OK] Dataset raw guardado en {RAW_PATH}")

Leyendo CSV desde /app/data/raw/ml_dataset.csv...


26/05/25 16:08:30 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Spark DataFrame cargado: 872,856 filas, 11 columnas
root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)



[OK] Dataset raw guardado en hdfs://hadoop:9000/raw/transactions.parquet


## 5. Transformaciones y limpieza de datos

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when

# Leer desde HDFS
sdf = spark.read.parquet(RAW_PATH)

# Columnas críticas en PaySim
critical_cols = ["amount", "type", "isFraud"]
sdf_clean = sdf.dropna(subset=[c for c in critical_cols if c in sdf.columns])

# Normalizar tipo de transacción a mayúsculas
sdf_clean = sdf_clean.withColumn("type", F.upper(F.trim(col("type"))))

# Filtrar importes negativos o cero
sdf_clean = sdf_clean.filter(col("amount") > 0)

print(f"Filas tras limpieza: {sdf_clean.count():,}")
sdf_clean.show(3, truncate=50)

Filas tras limpieza: 872,855
+----+--------+-------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type| amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+-------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT|9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT|1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|  181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
+----+--------+-------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
only showing top 3 rows



## 6. Feature engineering para el modelo de fraude

In [6]:
from pyspark.sql.functions import lit

# One-hot encoding del tipo de transacción
tipos = ["CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"]
sdf_features = sdf_clean

for t in tipos:
    col_name = f"type_{t}"
    sdf_features = sdf_features.withColumn(
        col_name,
        when(col("type") == t, 1.0).otherwise(0.0)
    )

# Renombrar columnas para la API del modelo
rename_map = {
    "oldbalanceOrg":  "old_balance_orig",
    "newbalanceOrig": "new_balance_orig",
    "oldbalanceDest": "old_balance_dest",
    "newbalanceDest": "new_balance_dest",
    "isFraud":        "label"
}
for old_name, new_name in rename_map.items():
    if old_name in sdf_features.columns:
        sdf_features = sdf_features.withColumnRenamed(old_name, new_name)

# YA TENEMOS LOS BALANCES (así que no los mockeamos),
# Pero seguimos mockeando a 0.0 los atributos de grafos (Neo4j) que nos exigiría el modelo.
for graph_col in ["orig_out_degree", "orig_pagerank", "orig_community",
                  "dest_in_degree", "dest_pagerank", "dest_community"]:
    if graph_col not in sdf_features.columns:
        sdf_features = sdf_features.withColumn(graph_col, lit(0.0))

print(f"Features preparadas: {len(sdf_features.columns)} columnas")
sdf_features.printSchema()

Features preparadas: 22 columnas
root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- old_balance_orig: double (nullable = true)
 |-- new_balance_orig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- old_balance_dest: double (nullable = true)
 |-- new_balance_dest: double (nullable = true)
 |-- label: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)
 |-- type_CASH_IN: double (nullable = false)
 |-- type_CASH_OUT: double (nullable = false)
 |-- type_DEBIT: double (nullable = false)
 |-- type_PAYMENT: double (nullable = false)
 |-- type_TRANSFER: double (nullable = false)
 |-- orig_out_degree: double (nullable = false)
 |-- orig_pagerank: double (nullable = false)
 |-- orig_community: double (nullable = false)
 |-- dest_in_degree: double (nullable = false)
 |-- dest_pagerank: double (nullable = false)
 |-- dest_community: double (nullab

## 7. Guardar datos procesados en HDFS

In [7]:
import os

PROCESSED_PATH = f"{DATA_ROOT}/data/processed/transactions_clean.parquet"

if not HDFS_UP:
    os.makedirs(f"{LOCAL_DATA}/data/processed", exist_ok=True)

sdf_features.write.mode("overwrite").parquet(PROCESSED_PATH)
print(f"[OK] Datos procesados guardados en {PROCESSED_PATH}")

# Verificar uso de espacio usando la API Java de Spark (no requiere cliente hdfs instalado)
if HDFS_UP:
    try:
        fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
        summary = fs.getContentSummary(spark._jvm.org.apache.hadoop.fs.Path("/data/"))
        size_mb = summary.getLength() / 1024 / 1024
        print(f"Uso de espacio HDFS /data/: {size_mb:.1f} MB")
    except Exception as e:
        print(f"[WARN] No se pudo consultar espacio HDFS: {e}")


[OK] Datos procesados guardados en hdfs://hadoop:9000/data/processed/transactions_clean.parquet
Uso de espacio HDFS /data/: 36.1 MB


## 8. Verificación final del estado de HDFS

In [8]:
import os

print(f"=== Estructura de datos ({DATA_ROOT}) ===")

if HDFS_UP:
    # Listar usando la API Java de HDFS
    fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
    for path in ["/data/raw", "/data/processed", "/data/fraud-results"]:
        try:
            files = fs.listStatus(spark._jvm.org.apache.hadoop.fs.Path(path))
            print(f"\n{path}/")
            for f in files:
                size_mb = f.getLen() / 1024 / 1024
                print(f"  {f.getPath().getName()}  ({size_mb:.2f} MB)")
        except Exception as e:
            print(f"\n{path}/ → no existe aún o error: {e}")
else:
    # Listar en sistema de archivos local
    for subdir in ["raw", "processed", "fraud-results"]:
        path = os.path.join(LOCAL_DATA, "data", subdir)
        if os.path.exists(path):
            print(f"\n{path}/")
            for f in os.listdir(path):
                size_mb = os.path.getsize(os.path.join(path, f)) / 1024 / 1024
                print(f"  {f}  ({size_mb:.2f} MB)")
        else:
            print(f"\n{path}/ → no existe aún")

spark.stop()
print("\nSparkSession cerrada. Ingesta completada.")


=== Estructura de datos (hdfs://hadoop:9000) ===

/data/raw/

/data/processed/
  transactions_clean.parquet  (0.00 MB)

/data/fraud-results/
  batch_predictions.parquet  (0.00 MB)

SparkSession cerrada. Ingesta completada.
